# Deep Learning Assignment — Practice 2: RNNs

| | |
|---|---|
| **Students** | Gianluca Lascaro, Raffaele Rizzuti |
| **University** | Universidad de A Coruña |
| **Year** | 2025–2026 |

## 1. Data Preprocessing
The data preparation follows these steps:
1. **Cleaning**: Remove rows from 2026.
2. **Splitting**: Train (2021-2023), Val (2024), and Test (2025) using a chronological split.
3. **Normalization**: Scaling based on training mean and standard deviation.
4. **Sequencing**: Creating sliding windows for time-series forecasting.

In [ ]:
import numpy as np
import pandas as pd

# Load and clean data
df = pd.read_csv("nyiso_hourly_load.csv", parse_dates=["Time Stamp"])
df = df[df["Time Stamp"].dt.year < 2026].copy()

# Temporal split
FEATURE_COLS = [c for c in df.columns if c != "Time Stamp"]
TARGET_COL = "total_load"

train = df[df["Time Stamp"].dt.year <= 2023].copy()
val = df[df["Time Stamp"].dt.year == 2024].copy()
test = df[df["Time Stamp"].dt.year == 2025].copy()

In [ ]:
# Normalization (Z-score scaling)
mean = train[FEATURE_COLS].mean()
std = train[FEATURE_COLS].std()

train_scaled = train[FEATURE_COLS].sub(mean).div(std)
val_scaled = val[FEATURE_COLS].sub(mean).div(std)
test_scaled = test[FEATURE_COLS].sub(mean).div(std)

In [ ]:
def make_sequences(data, seq_len=24, horizon=3, target_idx=-1):
    """Creates X (sequences) and y (target at t + horizon)."""
    X, y = [], []
    for i in range(len(data) - seq_len - horizon + 1):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len + horizon - 1, target_idx])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

SEQ_LEN = 24
X_train, y_train = make_sequences(train_scaled.values, SEQ_LEN)
X_val, y_val = make_sequences(val_scaled.values, SEQ_LEN)
X_test, y_test = make_sequences(test_scaled.values, SEQ_LEN)

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Val:   {X_val.shape}, {y_val.shape}")
print(f"Test:  {X_test.shape}, {y_test.shape}")